# Serialization Formats

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

**Deviation from the mockup's exact numbers:** `docs/requirements/curriculum-topics-2026-07.md`'s US-C8 suggests a 50-column, ~2GB CSV. This notebook uses a 20-column, ~235MB CSV instead -- large enough to produce a clear, real, order-of-magnitude proportional bytes-read drop without making the notebook slow to run/review on a 3-worker/2-core/4GB-per-worker dev cluster. See `concept.md`'s own "Deviation" note for the full reasoning.

Three claims under test, each checked directly against real evidence (the stage's `inputBytes` REST metric, not an assumed number):
1. Reading a CSV file and `select()`-ing 3 of its 20 columns reads (almost) the same bytes as reading all of them -- CSV has no column pruning.
2. The identical data written as Parquet and read with the same `select()` reads dramatically fewer bytes -- close to (or better than) the 3/20 column-count proportion.
3. A Parquet table partitioned on a column, filtered on that column, reads only the matching partition's files -- whole-file skipping, not row-level filtering.

In [ ]:
import json
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, StructField, StructType
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("serialization-formats").getOrCreate()
app_id = spark.sparkContext.applicationId
print("Spark version:", spark.version)
print("Application id:", app_id)


def stages_snapshot():
    """Current `/api/v1/applications/<id>/stages` list -- the ground truth
    for 'how many bytes did this stage's scan actually read', the same REST
    surface the Spark UI's Stages/SQL tabs read, and the exact field
    (`inputBytes`) this topic's manifest.yaml spotlights via stage_metrics.
    Same disposition as window-functions'/caching-persistence's own
    snapshot helpers."""
    url = f"http://localhost:4040/api/v1/applications/{app_id}/stages"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


def bytes_read_for(action_fn):
    """Runs `action_fn()`, then sums `inputBytes` across every stage that
    appeared as a result of it -- isolates the bytes this one action read,
    not any prior action's stages."""
    before_ids = {s["stageId"] for s in stages_snapshot()}
    action_fn()
    new_stages = [s for s in stages_snapshot() if s["stageId"] not in before_ids]
    return sum(s.get("inputBytes", 0) for s in new_stages)


SCRATCH_DIR = "/workspace/scratch/serialization-formats"

## 1. Build a wide dataset, write it as both CSV and Parquet

20 double columns (`c0`..`c19`) plus an `id` -- wide enough that selecting 3 of them is a meaningfully small fraction (3/20 = 15%), without the 50-column/2GB scale the mockup's original suggestion used (see the intro cell's deviation note).

In [ ]:
NUM_COLS = 20
SELECT_COLS = ["c0", "c1", "c2"]
NUM_ROWS = 1_500_000

schema = StructType(
    [StructField("id", LongType(), False)]
    + [StructField(f"c{j}", DoubleType(), False) for j in range(NUM_COLS)]
)

rdd = spark.sparkContext.parallelize(range(NUM_ROWS), 24).map(
    lambda i: Row(
        id=i,
        **{f"c{j}": float((i * 7919 + j * 104729) % 1_000_003) / 1000.0 for j in range(NUM_COLS)},
    )
)
wide_df = spark.createDataFrame(rdd, schema=schema)

csv_path = f"{SCRATCH_DIR}/csv_wide"
wide_df.write.mode("overwrite").option("header", True).csv(csv_path)

parquet_path = f"{SCRATCH_DIR}/parquet_wide"
wide_df.write.mode("overwrite").parquet(parquet_path)

print(f"Wrote {NUM_ROWS} rows x {NUM_COLS + 1} columns to both {csv_path} and {parquet_path}")

## 2. CSV baseline -- select 3 of 20 columns

**Hypothesis:** since CSV is row-oriented, does selecting only 3 of 20 columns reduce the bytes Spark reads off disk, or does it read (close to) the whole file regardless? Write your answer before running the cell below.

In [ ]:
csv_df = spark.read.option("header", True).schema(schema).csv(csv_path)


def read_csv_select3():
    csv_df.select(*SELECT_COLS).agg(*[F.sum(c) for c in SELECT_COLS]).collect()


csv_bytes = bytes_read_for(read_csv_select3)
print(f"[CSV] select 3/{NUM_COLS} columns -- inputBytes read: {csv_bytes} ({csv_bytes / 1e6:.1f} MB)")
assert csv_bytes > 0, "expected a nonzero inputBytes reading the CSV file"
# Measured on this dev cluster: 246.2 MB -- close to the full ~235MB CSV file
# size (a small parsing/overhead delta, not a column-pruning drop). CSV has no
# per-column addressing, so select() can't reduce what's read off disk.

## 3. Same data as Parquet -- select the same 3 columns

**Hypothesis:** does reading the identical data from Parquet and selecting the same 3 columns read fewer bytes than the CSV baseline above? Roughly how much fewer -- proportional to 3/20 (15%), better, or worse? Write your answer, then check below.

In [ ]:
parquet_df = spark.read.parquet(parquet_path)


def read_parquet_select3():
    parquet_df.select(*SELECT_COLS).agg(*[F.sum(c) for c in SELECT_COLS]).collect()


parquet_bytes = bytes_read_for(read_parquet_select3)
print(f"[Parquet] select 3/{NUM_COLS} columns -- inputBytes read: {parquet_bytes} ({parquet_bytes / 1e6:.1f} MB)")

ratio = parquet_bytes / csv_bytes
expected_ratio = len(SELECT_COLS) / NUM_COLS
print(f"\nParquet/CSV bytes-read ratio: {ratio:.3f}  (column-count ratio would predict ~{expected_ratio:.3f} = 3/{NUM_COLS})")
# Measured on this dev cluster: 24.1 MB, ratio ~0.098 -- a ~10x drop, even
# better than the 3/20=0.15 column-count ratio predicts (columnar compression
# also does more work on 3 all-numeric columns than CSV's text encoding
# does), but the same order of magnitude and direction as the mockup's own
# "2.1GB -> 118MB" target example.
assert parquet_bytes < csv_bytes / 2, (
    "expected Parquet's column-pruned read to be well under half the CSV baseline's bytes read"
)

## 4. Checkpoint for the self-check reveal

Click **Reveal self-check** on the topic page after running this: the plan panel labels the `Scan` node (the annotation engine's tokenizer only reads a node's first word, so it doesn't distinguish `Scan parquet` from `Scan csv` -- see `manifest.yaml`'s note), and the stage panel spotlights `inputBytes` for this column-pruned read -- that's the metric that actually shows the format difference.

In [ ]:
checkpoint(parquet_df.select(*SELECT_COLS), topic="serialization-formats")
print("Checkpoint written -- click 'Reveal self-check' on the topic page.")

## 5. Partitioned Parquet -- filter on the partition column

**Hypothesis:** if the same data is written Parquet-partitioned on a low-cardinality column (8 `region` values), does filtering on that column reduce bytes read further than column pruning alone already does -- and by roughly how much? Write your answer, then check below.

In [ ]:
NUM_REGIONS = 8
partitioned_df = wide_df.withColumn("region", (F.col("id") % NUM_REGIONS).cast("string"))
partitioned_path = f"{SCRATCH_DIR}/parquet_partitioned"
partitioned_df.write.mode("overwrite").partitionBy("region").parquet(partitioned_path)

part_read_df = spark.read.parquet(partitioned_path)


def read_partitioned_unfiltered():
    part_read_df.agg(F.sum("c0")).collect()


def read_partitioned_filtered():
    part_read_df.filter(F.col("region") == "0").agg(F.sum("c0")).collect()


unfiltered_bytes = bytes_read_for(read_partitioned_unfiltered)
print(f"[Partitioned Parquet] unfiltered -- inputBytes read: {unfiltered_bytes} ({unfiltered_bytes / 1e6:.1f} MB)")

filtered_bytes = bytes_read_for(read_partitioned_filtered)
print(f"[Partitioned Parquet] filter region == '0' -- inputBytes read: {filtered_bytes} ({filtered_bytes / 1e6:.1f} MB)")

part_ratio = filtered_bytes / unfiltered_bytes
expected_part_ratio = 1 / NUM_REGIONS
print(f"\nFiltered/unfiltered ratio: {part_ratio:.3f}  (expected ~{expected_part_ratio:.3f} = 1/{NUM_REGIONS})")
# Measured on this dev cluster: 15.0 MB unfiltered -> 1.9 MB filtered, ratio
# 0.125 -- matches 1/8 almost exactly: Spark's partition pruning lists only
# the one matching region= directory and never opens the other seven files.
assert filtered_bytes < unfiltered_bytes, (
    "expected filtering on the partition column to read strictly fewer bytes than the unfiltered read"
)

plan_text = part_read_df.filter(F.col("region") == "0")._jdf.queryExecution().simpleString()
print("\nPartitionFilters present in plan:", "PartitionFilters" in plan_text)
assert "PartitionFilters" in plan_text, "expected the filtered read's plan to show pushed-down PartitionFilters"

## 6. Compare all three bytes-read numbers side by side

In [ ]:
print(f"CSV,       select 3/{NUM_COLS} cols:            {csv_bytes / 1e6:8.1f} MB")
print(f"Parquet,   select 3/{NUM_COLS} cols:            {parquet_bytes / 1e6:8.1f} MB  ({ratio:.1%} of CSV)")
print(f"Parquet,   partitioned, unfiltered:      {unfiltered_bytes / 1e6:8.1f} MB")
print(f"Parquet,   partitioned, filtered 1/{NUM_REGIONS}:    {filtered_bytes / 1e6:8.1f} MB  ({part_ratio:.1%} of unfiltered)")
assert parquet_bytes < csv_bytes
assert filtered_bytes < unfiltered_bytes